In [4]:
# IMPORTANT: Always run cells in order from top to bottom
# after restarting the kernel. The kernel has no memory
# of previous runs. df must be created before it can be cleaned.

import pandas as pd

# Load the dwelling built file
# skiprows=9 tells pandas to ignore the SuperWEB2 metadata at the top
df = pd.read_csv('../data/raw/GHS2022_dwelling_built.csv', skiprows=9)

# Question 1 from our cheat sheet - what shape is this data?
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Shape: (101, 5)

Column names:
['Summation Options', 'Province', 'Dwelling originally built', 'Count', 'Unnamed: 4']


In [3]:
# Let us look at the actual data before touching anything
print("First 15 rows:")
print(df.head(15))
print("\nLast 5 rows:")
print(df.tail(5))
print("\nUnnamed column - any values in it:")
print(df['Unnamed: 4'].value_counts())
print("\nHow many nulls in each column:")
print(df.isnull().sum())

First 15 rows:
   Summation Options      Province Dwelling originally built         Count  \
0      Person Weight  Western Cape     2018–2022 (0–5 years)  2.490587e+05   
1      Person Weight  Western Cape    2013–2017 (6–10 years)  4.997046e+05   
2      Person Weight  Western Cape   2003–2012 (11–20 years)  1.075748e+06   
3      Person Weight  Western Cape   1993–2002 (21–30 years)  1.228291e+06   
4      Person Weight  Western Cape   1983–1992 (31–40 years)  1.185208e+06   
5      Person Weight  Western Cape   1973–1982 (41–50 years)  8.193509e+05   
6      Person Weight  Western Cape   1951–1973 (51–70 years)  4.911722e+05   
7      Person Weight  Western Cape             Prior to 1951  1.475377e+05   
8      Person Weight  Western Cape               DO NOT KNOW  1.534787e+06   
9      Person Weight  Western Cape                     Total  7.230858e+06   
10     Person Weight  Eastern Cape     2018–2022 (0–5 years)  1.608099e+05   
11     Person Weight  Eastern Cape    2013–2017 (

In [5]:
# Check current state of the DataFrame
print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())

Shape: (101, 5)

Column names: ['Summation Options', 'Province', 'Dwelling originally built', 'Count', 'Unnamed: 4']

Dtypes:
Summation Options             object
Province                      object
Dwelling originally built     object
Count                        float64
Unnamed: 4                   float64
dtype: object

First 5 rows:
  Summation Options      Province Dwelling originally built         Count  \
0     Person Weight  Western Cape     2018–2022 (0–5 years)  2.490587e+05   
1     Person Weight  Western Cape    2013–2017 (6–10 years)  4.997046e+05   
2     Person Weight  Western Cape   2003–2012 (11–20 years)  1.075748e+06   
3     Person Weight  Western Cape   1993–2002 (21–30 years)  1.228291e+06   
4     Person Weight  Western Cape   1983–1992 (31–40 years)  1.185208e+06   

   Unnamed: 4  
0         NaN  
1         NaN  
2         NaN  
3         NaN  
4         NaN  


In [3]:
# ============================================
# FULL CLEANING — GHS2022_dwelling_built
# ============================================

# Step 1 — Drop useless columns
df = df.drop(columns=['Summation Options', 'Unnamed: 4'])

# Step 2 — Drop Total rows and copyright line
df = df[df['Province'] != 'Total']
df = df[df['Dwelling originally built'] != 'Total']
df = df.dropna()

# Step 3 — Rename columns
df = df.rename(columns={
    'Province': 'province',
    'Dwelling originally built': 'dwelling_period',
    'Count': 'count'
})

# Step 4 — Fix count dtype from float to int
df['count'] = df['count'].round(0).astype(int)

# Step 5 — Reset index
df = df.reset_index(drop=True)

# ============================================
# VERIFICATION
# ============================================

print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nFirst 10 rows:")
print(df.head(10))
print("\nUnique provinces:", df['province'].unique())
print("\nUnique periods:", df['dwelling_period'].unique())
print("\nAny nulls?", df.isnull().sum().sum())

Shape: (81, 3)

Dtypes:
province             str
dwelling_period      str
count              int64
dtype: object

First 10 rows:
       province          dwelling_period    count
0  Western Cape    2018–2022 (0–5 years)   249059
1  Western Cape   2013–2017 (6–10 years)   499705
2  Western Cape  2003–2012 (11–20 years)  1075748
3  Western Cape  1993–2002 (21–30 years)  1228291
4  Western Cape  1983–1992 (31–40 years)  1185208
5  Western Cape  1973–1982 (41–50 years)   819351
6  Western Cape  1951–1973 (51–70 years)   491172
7  Western Cape            Prior to 1951   147538
8  Western Cape              DO NOT KNOW  1534787
9  Eastern Cape    2018–2022 (0–5 years)   160810

Unique provinces: <StringArray>
[ 'Western Cape',  'Eastern Cape', 'Northern Cape',    'Free State',
 'KwaZulu-Natal',    'North West',       'Gauteng',    'Mpumalanga',
       'Limpopo']
Length: 9, dtype: str

Unique periods: <StringArray>
[  '2018–2022 (0–5 years)',  '2013–2017 (6–10 years)',
 '2003–2012 (11–20 years

In [4]:
# ============================================
# SAVE CLEANED FILE
# ============================================
df.to_csv('../data/cleaned/GHS2022_dwelling_built_cleaned.csv', index=False)

print("Saved successfully.")
print("Clean file location: data/cleaned/GHS2022_dwelling_built_cleaned.csv")

Saved successfully.
Clean file location: data/cleaned/GHS2022_dwelling_built_cleaned.csv


In [2]:
def clean_ghs_file(filename, value_column_name):
    """
    Cleans a SuperWEB2 exported GHS2022 CSV file.
    Handles three export formats from SuperWEB2.
    """
    with open(f'../data/raw/{filename}', 'r') as f:
        lines = f.readlines()
    
    # Get the first data line to detect format
    first_data_line = lines[10].strip()
    
    # Format detection
    # Format 1 - Standard: no quotes on first line
    # Format 2 - Double wrapped: entire line in one quote with ""value""
    # Format 3 - Standard CSV with quotes: "value","value","value"
    
    if first_data_line.startswith('"') and '""' in first_data_line:
        # Format 2 - double wrapped (electricity style)
        format_type = 'double_wrapped'
    elif first_data_line.startswith('"') and '""' not in first_data_line:
        # Format 3 - standard CSV with quotes (dwelling_type style)
        format_type = 'standard_quoted'
    else:
        # Format 1 - clean standard (dwelling_built style)
        format_type = 'standard'
    
    print(f"Loading {filename} | Format: {format_type}")
    
    if format_type == 'double_wrapped':
        rows = []
        for line in lines[9:]:
            line = line.strip()
            if not line or 'Copyright' in line:
                continue
            if line.startswith('"') and line.endswith('"'):
                line = line[1:-1]
            line = line.replace('""', '<<QUOTE>>')
            parts = line.split(',<<QUOTE>>')
            parts = [p.replace('<<QUOTE>>', '').strip(',').strip('"')
                    for p in parts]
            parts = [p for p in parts if p]
            if len(parts) >= 4:
                rows.append(parts[:4])
            elif len(parts) == 3:
                last = parts[-1]
                last_comma = last.rfind(',')
                if last_comma > 0:
                    fixed = parts[:-1] + [
                        last[:last_comma],
                        last[last_comma+1:]
                    ]
                    rows.append(fixed[:4])
        
        df = pd.DataFrame(rows, columns=[
            'Summation Options',
            'Province',
            value_column_name,
            'count'
        ])
        df['count'] = pd.to_numeric(df['count'], errors='coerce')
    
    elif format_type == 'standard_quoted':
        # Pandas handles this natively with quotechar parameter
        df = pd.read_csv(
            f'../data/raw/{filename}',
            skiprows=9,
            quotechar='"',
            skipinitialspace=True
        )
        cols = df.columns.tolist()
        # Drop ghost column if present
        df = df.drop(columns=[c for c in cols if 'Unnamed' in str(c)],
                    errors='ignore')
        cols = df.columns.tolist()
        df = df.rename(columns={
            cols[0]: 'Summation Options',
            cols[1]: 'Province',
            cols[2]: value_column_name,
            cols[3]: 'count'
        })
    
    else:
        # Standard format
        df = pd.read_csv(f'../data/raw/{filename}', skiprows=9)
        cols = df.columns.tolist()
        df = df.drop(columns=[c for c in cols if 'Unnamed' in str(c)],
                    errors='ignore')
        cols = df.columns.tolist()
        df = df.rename(columns={
            cols[0]: 'Summation Options',
            cols[1]: 'Province',
            cols[2]: value_column_name,
            cols[3]: 'count'
        })

    # Shared cleaning for all formats
    df = df[df['Province'] != 'Total']
    df = df[df[value_column_name] != 'Total']
    
    if 'Summation Options' in df.columns:
        df = df.drop(columns=['Summation Options'])
    
    df = df.rename(columns={'Province': 'province'})
    df = df.dropna()
    df['count'] = pd.to_numeric(df['count'], errors='coerce')
    df['count'] = df['count'].round(0).astype(int)
    df = df.reset_index(drop=True)
    
    output_name = filename.replace('.csv', '_cleaned.csv')
    df.to_csv(f'../data/cleaned/{output_name}', index=False)
    
    print(f"Cleaned shape: {df.shape}")
    print(f"Provinces: {df['province'].nunique()}")
    print(f"Nulls: {df.isnull().sum().sum()}")
    print(f"Saved: data/cleaned/{output_name}")
    print("---")
    
    return df

In [12]:
# Clean electricity main source
df_electricity = clean_ghs_file(
    'GHS2022_electricity_main_source.csv',
    'electricity_source'
)

df_electricity.head()

Loading GHS2022_electricity_main_source.csv
Raw shape after parse: (91, 4)
Cleaned shape: (72, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_electricity_main_source_cleaned.csv
---


,province,electricity_source,count
0,Western Cape,In-house pre-paid meter,6231196
1,Western Cape,In-house conventional meter,637688
2,Western Cape,Connected to other source which the household ...,203778
3,Western Cape,Connected to other source which the household ...,50850
4,Western Cape,Generator,0


In [16]:
# Clean all remaining files
files = [
    ('GHS2022_dwelling_type.csv', 'dwelling_type'),
    ('GHS2022_electricity_access.csv', 'electricity_access'),
    ('GHS2022_handwashing.csv', 'handwashing_facility'),
    ('GHS2022_refuse_irregular.csv', 'refuse_irregular'),
    ('GHS2022_refuse_removal.csv', 'refuse_type'),
    ('GHS2022_toilet_distance.csv', 'toilet_distance'),
    ('GHS2022_toilet_type.csv', 'toilet_type'),
    ('GHS2022_water_distance.csv', 'water_distance'),
    ('GHS2022_water_main_source.csv', 'water_source'),
    ('GHS2022_water_supply.csv', 'water_supply'),
]

results = {}

for filename, column_name in files:
    df_temp = clean_ghs_file(filename, column_name)
    results[column_name] = df_temp

print("All files processed.")
print(f"Total datasets cleaned: {len(results)}")

Loading GHS2022_dwelling_type.csv | Format: standard_quoted
Cleaned shape: (108, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_dwelling_type_cleaned.csv
---
Loading GHS2022_electricity_access.csv | Format: standard_quoted
Cleaned shape: (27, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_electricity_access_cleaned.csv
---
Loading GHS2022_handwashing.csv | Format: standard_quoted
Cleaned shape: (45, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_handwashing_cleaned.csv
---
Loading GHS2022_refuse_irregular.csv | Format: standard_quoted
Cleaned shape: (27, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_refuse_irregular_cleaned.csv
---
Loading GHS2022_refuse_removal.csv | Format: standard_quoted
Cleaned shape: (108, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_refuse_removal_cleaned.csv
---
Loading GHS2022_toilet_distance.csv | Format: standard_quoted
Cleaned shape: (54, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_toilet_distance_cleaned.csv
---
Lo

In [17]:
# ============================================
# CLEANING SUMMARY
# ============================================
import os

cleaned_files = os.listdir('../data/cleaned/')
print(f"Total cleaned files: {len(cleaned_files)}")
print("\nFiles in data/cleaned/:")
for f in sorted(cleaned_files):
    df_check = pd.read_csv(f'../data/cleaned/{f}')
    print(f"  {f}: {df_check.shape} | "
          f"Provinces: {df_check['province'].nunique()}")

Total cleaned files: 12

Files in data/cleaned/:
  GHS2022_dwelling_built_cleaned.csv: (81, 3) | Provinces: 9
  GHS2022_dwelling_type_cleaned.csv: (108, 3) | Provinces: 9
  GHS2022_electricity_access_cleaned.csv: (27, 3) | Provinces: 9
  GHS2022_electricity_main_source_cleaned.csv: (72, 3) | Provinces: 9
  GHS2022_handwashing_cleaned.csv: (45, 3) | Provinces: 9
  GHS2022_refuse_irregular_cleaned.csv: (27, 3) | Provinces: 9
  GHS2022_refuse_removal_cleaned.csv: (108, 3) | Provinces: 9
  GHS2022_toilet_distance_cleaned.csv: (54, 3) | Provinces: 9
  GHS2022_toilet_type_cleaned.csv: (117, 3) | Provinces: 9
  GHS2022_water_distance_cleaned.csv: (54, 3) | Provinces: 9
  GHS2022_water_main_source_cleaned.csv: (126, 3) | Provinces: 9
  GHS2022_water_supply_cleaned.csv: (36, 3) | Provinces: 9


In [5]:
# Clean littering dataset
df_littering = clean_ghs_file(
    'GHS2022_littering.csv',
    'littering'
)

df_littering.head(15)

Loading GHS2022_littering.csv | Format: standard_quoted
Cleaned shape: (27, 3)
Provinces: 9
Nulls: 0
Saved: data/cleaned/GHS2022_littering_cleaned.csv
---


,province,littering,count
0,Western Cape,Yes,1898658
1,Western Cape,No,5332200
2,Western Cape,Unspecified,0
3,Eastern Cape,Yes,2672789
4,Eastern Cape,No,3866266
5,Eastern Cape,Unspecified,0
6,Northern Cape,Yes,586257
7,Northern Cape,No,707787
8,Northern Cape,Unspecified,0
9,Free State,Yes,1677910


In [6]:
# ============================================
# RELOAD ALL DATASETS
# ============================================

cleaned_path = '../data/cleaned/'

electricity_source = pd.read_csv(f'{cleaned_path}GHS2022_electricity_main_source_cleaned.csv')
electricity_access = pd.read_csv(f'{cleaned_path}GHS2022_electricity_access_cleaned.csv')
water_source = pd.read_csv(f'{cleaned_path}GHS2022_water_main_source_cleaned.csv')
water_supply = pd.read_csv(f'{cleaned_path}GHS2022_water_supply_cleaned.csv')
water_distance = pd.read_csv(f'{cleaned_path}GHS2022_water_distance_cleaned.csv')
toilet_type = pd.read_csv(f'{cleaned_path}GHS2022_toilet_type_cleaned.csv')
toilet_distance = pd.read_csv(f'{cleaned_path}GHS2022_toilet_distance_cleaned.csv')
refuse_removal = pd.read_csv(f'{cleaned_path}GHS2022_refuse_removal_cleaned.csv')
refuse_irregular = pd.read_csv(f'{cleaned_path}GHS2022_refuse_irregular_cleaned.csv')
handwashing = pd.read_csv(f'{cleaned_path}GHS2022_handwashing_cleaned.csv')
dwelling_type = pd.read_csv(f'{cleaned_path}GHS2022_dwelling_type_cleaned.csv')
dwelling_built = pd.read_csv(f'{cleaned_path}GHS2022_dwelling_built_cleaned.csv')
littering = pd.read_csv(f'{cleaned_path}GHS2022_littering_cleaned.csv')

print("All datasets reloaded:")
for name, df in {
    'electricity_source': electricity_source,
    'electricity_access': electricity_access,
    'water_source': water_source,
    'water_supply': water_supply,
    'water_distance': water_distance,
    'toilet_type': toilet_type,
    'toilet_distance': toilet_distance,
    'refuse_removal': refuse_removal,
    'refuse_irregular': refuse_irregular,
    'handwashing': handwashing,
    'dwelling_type': dwelling_type,
    'dwelling_built': dwelling_built,
    'littering': littering
}.items():
    print(f"  {name}: {df.shape}")

All datasets reloaded:
  electricity_source: (72, 3)
  electricity_access: (27, 3)
  water_source: (126, 3)
  water_supply: (36, 3)
  water_distance: (54, 3)
  toilet_type: (117, 3)
  toilet_distance: (54, 3)
  refuse_removal: (108, 3)
  refuse_irregular: (27, 3)
  handwashing: (45, 3)
  dwelling_type: (108, 3)
  dwelling_built: (81, 3)
  littering: (27, 3)


In [7]:
# ============================================
# WATER — EXPLORE CATEGORIES
# ============================================

print("WATER SUPPLY categories:")
print(water_supply['water_supply'].unique())
print(f"Shape: {water_supply.shape}")

print("\nWATER DISTANCE categories:")
print(water_distance['water_distance'].unique())
print(f"Shape: {water_distance.shape}")

print("\nWATER SOURCE categories:")
print(water_source['water_source'].unique())
print(f"Shape: {water_source.shape}")

WATER SUPPLY categories:
<StringArray>
['Yes', 'No', 'Do not know', 'Unspecified']
Length: 4, dtype: str
Shape: (36, 3)

WATER DISTANCE categories:
<StringArray>
[          '201â€“500 metres', '501 metres â€“ 1 kilometre',
      'More than 1 kilometre',                'DO NOT KNOW',
             'Not applicable',       'Less than 200 metres']
Length: 6, dtype: str
Shape: (54, 3)

WATER SOURCE categories:
<StringArray>
[       'Piped (tap) water in dwelling',
 'Piped (tap) water on site or in yard',
                     'Borehole on site',
              'Rain-water tank on site',
                      'Neighbour's tap',
                  'Public/communal tap',
                 'Water-carrier/tanker',
                         'Water vendor',
                'Borehole outside yard',
           'Flowing water/stream/river',
              'Stagnant water/dam/pool',
                                 'Well',
                               'Spring',
       'Other source of drinking water']
Leng

In [11]:
# Read the raw file and fix encoding at source
water_distance = pd.read_csv(
    '../data/cleaned/GHS2022_water_distance_cleaned.csv',
    encoding='utf-8-sig'
)

# Replace the corrupted em dash sequence directly
water_distance['water_distance'] = (
    water_distance['water_distance']
    .str.replace('\u00e2\u20ac\u201c', '–', regex=False)
    .str.replace('\u00e2\u20ac\u2122', "'", regex=False)
    .str.strip()
)

print("Categories:")
print(water_distance['water_distance'].unique())

Categories:
<StringArray>
[          '201–500 metres', '501 metres – 1 kilometre',
    'More than 1 kilometre',              'DO NOT KNOW',
           'Not applicable',     'Less than 200 metres']
Length: 6, dtype: str


In [12]:
water_distance = pd.read_csv(
    f'{cleaned_path}GHS2022_water_distance_cleaned.csv',
    encoding='utf-8-sig'
)

In [13]:
water_source = pd.read_csv(
    '../data/cleaned/GHS2022_water_main_source_cleaned.csv',
    encoding='utf-8-sig'
)

print("Fixed source categories:")
print(water_source['water_source'].unique())

Fixed source categories:
<StringArray>
[       'Piped (tap) water in dwelling',
 'Piped (tap) water on site or in yard',
                     'Borehole on site',
              'Rain-water tank on site',
                      'Neighbour's tap',
                  'Public/communal tap',
                 'Water-carrier/tanker',
                         'Water vendor',
                'Borehole outside yard',
           'Flowing water/stream/river',
              'Stagnant water/dam/pool',
                                 'Well',
                               'Spring',
       'Other source of drinking water']
Length: 14, dtype: str


In [14]:
# ============================================
# SANITATION — EXPLORE CATEGORIES
# ============================================

print("TOILET TYPE categories:")
print(toilet_type['toilet_type'].unique())
print(f"\nShape: {toilet_type.shape}")

print("\nTOILET DISTANCE categories:")
print(toilet_distance['toilet_distance'].unique())
print(f"\nShape: {toilet_distance.shape}")

TOILET TYPE categories:
<StringArray>
[                 'Flush toilet connected to a public sewerage system',
         'Flush toilet connected to a septic tank or conservancy tank',
 'Pour bucket-flush toilet connected to a septic tank (or septic pit)',
                                                     'Chemical toilet',
                            'Pit latrine/toilet with ventilation pipe',
                         'Pit latrine/toilet without ventilation pipe',
                                                       'Bucket toilet',
                                               'Portable flush toilet',
                                                   'Composting toilet',
                                          'Urine diversion dry toilet',
                   'Open defecation (e.g. no facilities; field; bush)',
                                               'Other toilet facility',
                                                         'Unspecified']
Length: 13, dtype: str

Sh

In [15]:
print("HANDWASHING categories:")
print(handwashing['handwashing_facility'].unique())
print(f"\nShape: {handwashing.shape}")
print(f"\nSample:")
print(handwashing.head(15))

HANDWASHING categories:
<StringArray>
['Yes', 'No', 'Do not know', 'Not applicable', 'Unspecified']
Length: 5, dtype: str

Shape: (45, 3)

Sample:
         province handwashing_facility    count
0    Western Cape                  Yes  5948933
1    Western Cape                   No  1246978
2    Western Cape          Do not know     9895
3    Western Cape       Not applicable    25052
4    Western Cape          Unspecified        0
5    Eastern Cape                  Yes  3574739
6    Eastern Cape                   No  2787625
7    Eastern Cape          Do not know    12905
8    Eastern Cape       Not applicable   163785
9    Eastern Cape          Unspecified        0
10  Northern Cape                  Yes   875866
11  Northern Cape                   No   365698
12  Northern Cape          Do not know        0
13  Northern Cape       Not applicable    52480
14  Northern Cape          Unspecified        0


In [16]:
# ============================================
# ELECTRICITY — EXPLORE CATEGORIES
# ============================================

print("ELECTRICITY ACCESS categories:")
print(electricity_access['electricity_access'].unique())
print(f"Shape: {electricity_access.shape}")

print("\nELECTRICITY MAIN SOURCE categories:")
print(electricity_source['electricity_source'].unique())
print(f"Shape: {electricity_source.shape}")

ELECTRICITY ACCESS categories:
<StringArray>
['Yes', 'No', 'Unspecified']
Length: 3, dtype: str
Shape: (27, 3)

ELECTRICITY MAIN SOURCE categories:
<StringArray>
[                                                                                           'In-house pre-paid meter',
                                                                                        'In-house conventional meter',
  'Connected to other source which the household pays for (e.g. connected to neighbour's line and paying neighbours)',
 'Connected to other source which the household does not pay for (e.g. connected to neighbour's line and not paying)',
                                                                                                          'Generator',
                                                                                                  'Home solar panels',
                                                                            'Other source of main electricity supply',
     

In [17]:
# ============================================
# HOUSING — EXPLORE CATEGORIES
# ============================================

print("DWELLING TYPE categories:")
print(dwelling_type['dwelling_type'].unique())
print(f"Shape: {dwelling_type.shape}")

print("\nDWELLING BUILT categories:")
print(dwelling_built['dwelling_period'].unique())
print(f"Shape: {dwelling_built.shape}")

DWELLING TYPE categories:
<StringArray>
[    'Dwelling/house or brick/concrete block structure on a separate stand or yard or on farm',
                            'Traditional dwelling/hut/structure made of traditional materials',
                                                       'Flat or apartment in a block of flats',
                                                                    'Cluster house in complex',
                                                 'Town house (semi-detached house in complex)',
                                                                         'Semi-detached house',
                                                        'Dwelling/house/flat/room in backyard',
                                                         'Informal dwelling/shack in backyard',
 'Informal dwelling/shack not in backyard; e.g. in an informal/squatter settlement or on farm',
              'Room/flatlet on a property or a larger dwelling servants' quarters/granny flat',
